In [1]:
# CHECK NAN DISTRIBUTION IN TUMOR SAMPLES
import os
import sys
import subprocess
from pathlib import Path

import polars as pl
import pandas as pd
import urllib.request
import zipfile
import shutil

# =========================
# USER PARAMETERS (EDIT ME)
# =========================
BETA_PARQUET_PATH  = "/kaggle/input/gse225845-parquet/GSE225845.parquet"
PHENO_PARQUET_PATH = "/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet"

SAMPLE_ID_COL = "id_tissue"
LABEL_COL     = "label"
AGE_COL       = "age_at_surgery"

# Normal + Adjacent only
KEEP_LABELS = [0, 1]

# GP-age repo and working folders
GP_AGE_REPO_URL = "https://github.com/mirivar/GP-age.git"
GP_AGE_DIR      = Path("./GP-age").resolve()
MODEL_DATA_DIR  = GP_AGE_DIR / "model_data"

WORK_DIR        = Path("./gp_age_run_GSE225845").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

METH_CSV_PATH = WORK_DIR / "gp_age_meth_GSE225845_normal_adj.csv"
AGES_CSV_PATH = WORK_DIR / "gp_age_ages_GSE225845_normal_adj.csv"
OUTPUT_DIR    = WORK_DIR / "gp_age_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_TYPE = "30"   # 10 / 30 / 80 / a / b / c


# =========================
# FUNCTION: download GP-age models from Zenodo
# =========================
def download_gp_age_models():
    print(">>> Downloading GP-age model files (Zenodo)...")

    ZENODO_URL = "https://zenodo.org/record/6420531/files/gp_age_model_files.zip?download=1"
    ZIP_PATH = WORK_DIR / "gp_age_models.zip"

    # Download
    urllib.request.urlretrieve(ZENODO_URL, ZIP_PATH)
    print(f"  -> Downloaded model zip: {ZIP_PATH}")

    # Extract
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(WORK_DIR / "models_extracted")

    print("  -> Extracted model files.")

    # Move model_data into GP-age/model_data/
    extracted_dir = WORK_DIR / "models_extracted" / "model_data"
    if not MODEL_DATA_DIR.exists():
        MODEL_DATA_DIR.mkdir(parents=True)

    for file in extracted_dir.iterdir():
        shutil.copy(file, MODEL_DATA_DIR)

    print(f"  -> Copied models to: {MODEL_DATA_DIR}")


# =========================
# 1. LOAD PHENO
# =========================
print(">>> Loading phenotype table...")

pheno_pl = pl.read_parquet(PHENO_PARQUET_PATH)
for col in (SAMPLE_ID_COL, LABEL_COL, AGE_COL):
    if col not in pheno_pl.columns:
        raise ValueError(f"Column '{col}' not found in pheno table.")

pheno_keep = (
    pheno_pl
    .filter(
        pl.col(LABEL_COL).is_in(KEEP_LABELS) &
        pl.col(AGE_COL).is_not_null()
    )
    .select([SAMPLE_ID_COL, LABEL_COL, AGE_COL])
)

print(f"  -> {pheno_keep.height} samples kept ({KEEP_LABELS}).")
sample_ids = pheno_keep[SAMPLE_ID_COL].to_list()


# =========================
# 2. LOAD BETA MATRIX
# =========================
print(">>> Loading beta matrix for selected samples...")

beta_lf = pl.scan_parquet(BETA_PARQUET_PATH)
if SAMPLE_ID_COL not in beta_lf.columns:
    raise ValueError(f"Column '{SAMPLE_ID_COL}' not found in beta parquet.")

beta_keep = (
    beta_lf
    .filter(pl.col(SAMPLE_ID_COL).is_in(sample_ids))
    .collect()
)

print(f"  -> beta matrix shape: {beta_keep.height} samples x {beta_keep.width} columns")


# =========================
# 3. BUILD CpG x SAMPLE MATRIX FOR GP-age
# =========================
print(">>> Building methylation matrix for GP-age...")

cpg_cols = [c for c in beta_keep.columns if isinstance(c, str) and c.startswith("cg")]
print(f"  -> {len(cpg_cols)} CpG columns found.")

beta_pd = beta_keep.to_pandas().set_index(SAMPLE_ID_COL)
beta_pd = beta_pd.loc[sample_ids, cpg_cols]

meth_df = beta_pd.T
meth_df.index.name = "CpG"
meth_df.to_csv(METH_CSV_PATH)

print(f"  -> methylation CSV written: {METH_CSV_PATH}")


# =========================
# 4. BUILD AGE CSV
# =========================
print(">>> Building ages CSV...")

pheno_pd = pheno_keep.to_pandas().set_index(SAMPLE_ID_COL).loc[sample_ids]

ages_df = pheno_pd[[AGE_COL]].reset_index()
ages_df.columns = ["sample_id", "age"]
ages_df.to_csv(AGES_CSV_PATH, index=False)

print(f"  -> ages CSV written: {AGES_CSV_PATH}")


# =========================
# 5. CLONE REPO + INSTALL MODELS
# =========================
print(">>> Ensuring GP-age repo and models exist...")

# Clone repo
if not GP_AGE_DIR.exists():
    print(f"  -> Cloning GP-age repo into {GP_AGE_DIR}")
    subprocess.run(["git", "clone", GP_AGE_REPO_URL, str(GP_AGE_DIR)], check=True)
else:
    print("  -> Repo already present.")

# Install Python dependencies
deps = ["matplotlib", "pandas", "scikit-learn", "GPy"]
print("  -> Installing GP-age python dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", *deps], check=True)

# Download models
if not MODEL_DATA_DIR.exists() or len(list(MODEL_DATA_DIR.glob("*"))) < 6:
    download_gp_age_models()
else:
    print("  -> Model files already installed.")


>>> Loading phenotype table...
  -> 253 samples kept ([0, 1]).
>>> Loading beta matrix for selected samples...


/tmp/ipykernel_47/282515244.py:101: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if SAMPLE_ID_COL not in beta_lf.columns:


  -> beta matrix shape: 253 samples x 750428 columns
>>> Building methylation matrix for GP-age...
  -> 747602 CpG columns found.
  -> methylation CSV written: /kaggle/working/gp_age_run_GSE225845/gp_age_meth_GSE225845_normal_adj.csv
>>> Building ages CSV...
  -> ages CSV written: /kaggle/working/gp_age_run_GSE225845/gp_age_ages_GSE225845_normal_adj.csv
>>> Ensuring GP-age repo and models exist...
  -> Cloning GP-age repo into /kaggle/working/GP-age


Cloning into '/kaggle/working/GP-age'...


  -> Installing GP-age python dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 36.2 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
  -> Model files already installed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
tsfresh 0.21.0 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.12.0 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
imbalanced-learn 0.13.0 requires scikit-learn<2,>=1.3.2, but you have scikit-learn 1.2.2 which is incompatible.
plotnine 0.14.5 requires matplotlib>=3.8.0, but you have matplotlib 3.7.2 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.2.2 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.


In [2]:
# =========================
# 6. RUN GP-age
# =========================
print(">>> Running GP-age prediction...")

predict_script = GP_AGE_DIR / "predict_age.py"
if not predict_script.exists():
    raise FileNotFoundError("predict_age.py not found!")

cmd = [
    sys.executable,
    str(predict_script),
    "-x", str(METH_CSV_PATH),
    "-y", str(AGES_CSV_PATH),
    "-m", MODEL_TYPE,
    "-o", str(OUTPUT_DIR)
]

print("  -> Command:", " ".join(cmd))
print(">>> Running GP-age inside its repository directory...")
subprocess.run(cmd, check=True, cwd=str(GP_AGE_DIR))


print("\n>>> GP-age finished successfully!")
print(f"Output directory: {OUTPUT_DIR}")


>>> Running GP-age prediction...
  -> Command: /usr/bin/python3 /kaggle/working/GP-age/predict_age.py -x /kaggle/working/gp_age_run_GSE225845/gp_age_meth_GSE225845_normal_adj.csv -y /kaggle/working/gp_age_run_GSE225845/gp_age_ages_GSE225845_normal_adj.csv -m 30 -o /kaggle/working/gp_age_run_GSE225845/gp_age_output
>>> Running GP-age inside its repository directory...


INFO:GP-age:Starting age prediction using GP-age (30_cpgs)
INFO:GP-age:Loading methylation data...
INFO:GP-age:Successfully loaded 253 samples
INFO:GP-age:Loading GP-age model...
INFO:GP-age:GP-age successfully loaded
INFO:GP-age:Imputing 1265 missing values
INFO:GP-age:Starting age prediction...
INFO:GP-age:Age prediction completed
INFO:GP-age:Predictions were saved to /kaggle/working/gp_age_run_GSE225845/gp_age_output/GP-age_30_cpgs_predictions.csv
INFO:GP-age:Calculating statistics
INFO:GP-age:Stats were saved to /kaggle/working/gp_age_run_GSE225845/gp_age_output/GP-age_30_cpgs_stats.csv
INFO:GP-age:Done!



>>> GP-age finished successfully!
Output directory: /kaggle/working/gp_age_run_GSE225845/gp_age_output


In [3]:
# ------------------------------------------------------------
# LOAD GP-AGE METHYLATION MATRIX FROM THE CSV WE GENERATED
# ------------------------------------------------------------

# This is the file created earlier by the GP-age export step
METH_CSV_PATH = "./gp_age_run_GSE225845/gp_age_meth_GSE225845_normal_adj.csv"

# Load CpG x sample matrix
meth_raw = pd.read_csv(METH_CSV_PATH, index_col=0)

# Convert back to sample x CpG (needed for CpG coverage test)
beta_df = meth_raw.T.copy()

print("Loaded methylation matrix:")
print(" - CpGs:", beta_df.shape[1])
print(" - Samples:", beta_df.shape[0])


Loaded methylation matrix:
 - CpGs: 747602
 - Samples: 253


In [4]:
import pandas as pd
from pathlib import Path

pred_file = Path("./gp_age_run_GSE225845/gp_age_output").glob("*.csv")

pred_file = list(pred_file)
print("CSV trovati nella cartella output:\n", pred_file)

# Apriamo il file principale
df = pd.read_csv(pred_file[0])

print("\n=== PRIME RIGHE ===")
print(df.head())

print("\n=== COLONNE ===")
print(df.columns.tolist())


CSV trovati nella cartella output:
 [PosixPath('gp_age_run_GSE225845/gp_age_output/GP-age_30_cpgs_stats.csv'), PosixPath('gp_age_run_GSE225845/gp_age_output/GP-age_30_cpgs_predictions.csv')]

=== PRIME RIGHE ===
  Unnamed: 0   stats
0       RMSE  10.654
1      MedAE   5.907
2     MeanAE   7.775

=== COLONNE ===
['Unnamed: 0', 'stats']


In [12]:
# ============================================================
# GP-AGE — Evaluation & CpG Coverage Report (Auto-detect columns)
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

MODEL_TYPE = "30"

# ------------------------------------------------------------
# 1. Load CpGs
# ------------------------------------------------------------
model_file = Path("./GP-age/model_data") / f"GP-age_sites_{MODEL_TYPE}_cpgs.csv"
gp_age_cpgs = pd.read_csv(model_file).iloc[:, 0].tolist()

print(f"GP-age CpGs required: {len(gp_age_cpgs)}")

# ------------------------------------------------------------
# 2. Load methylation matrix used for GP-age
# ------------------------------------------------------------
METH_CSV_PATH = Path("./gp_age_run_GSE225845/gp_age_meth_GSE225845_normal_adj.csv")
meth_raw = pd.read_csv(METH_CSV_PATH, index_col=0)

present_cpgs = set(meth_raw.index)
missing_cpgs = [c for c in gp_age_cpgs if c not in present_cpgs]

print("\n=== CPG COVERAGE ===")
print("Required:", len(gp_age_cpgs))
print("Present :", len(gp_age_cpgs) - len(missing_cpgs))
print("Missing :", len(missing_cpgs))

# ------------------------------------------------------------
# 3. LOAD GP-AGE PREDICTIONS (auto-detect columns)
# ------------------------------------------------------------

pred_file = Path("./gp_age_run_GSE225845/gp_age_output") / f"GP-age_{MODEL_TYPE}_cpgs_predictions.csv"
if not pred_file.exists():
    raise FileNotFoundError(f"Prediction file not found: {pred_file}")

gp_pred_df = pd.read_csv(pred_file)

print("\n=== CONTENT OF GP-age PREDICTION FILE ===")
print(gp_pred_df.head())
print("Columns:", gp_pred_df.columns.tolist())

# Identify sample column (first col)
sample_col = gp_pred_df.columns[0]

# Identify predicted age column (the one containing "age")
pred_cols = [c for c in gp_pred_df.columns if "age" in c.lower()]
if len(pred_cols) == 0:
    raise ValueError("No column containing 'age' found in prediction file.")

pred_age_col = pred_cols[-1]  # usually last column

print(f"\n-> Detected sample column:       {sample_col}")
print(f"-> Detected predicted age col:   {pred_age_col}")

# Rename for consistency
gp_pred_df = gp_pred_df.rename(columns={sample_col: "sample_id", pred_age_col: "GP_age"})

# ------------------------------------------------------------
# 4. MERGE WITH PHENOTYPE
# ------------------------------------------------------------

pheno = pheno_pl.to_pandas().copy()

if "id_tissue" not in pheno.columns:
    raise ValueError("Column 'id_tissue' missing in pheno table.")

pheno = pheno.merge(
    gp_pred_df[["sample_id", "GP_age"]],
    left_on="id_tissue",
    right_on="sample_id",
    how="inner"
)

print(f"\nRows after merge: {len(pheno)}")

# ------------------------------------------------------------
# 5. BUILD EVAL DATASET
# ------------------------------------------------------------

eval_df = (
    pheno[pheno["label"].isin([0,1])]
    .dropna(subset=["age_at_surgery", "GP_age"])
    .copy()
)

print("Samples used for evaluation:", len(eval_df))

y_true = eval_df["age_at_surgery"].values
y_pred = eval_df["GP_age"].values

# ------------------------------------------------------------
# 6. METRICS
# ------------------------------------------------------------

def RMSE(y,yhat): return np.sqrt(np.mean((y-yhat)**2))
def MAE(y,yhat): return np.mean(np.abs(y-yhat))
def MAPE(y,yhat): return np.mean(np.abs((y-yhat)/y))*100

global_mae  = MAE(y_true, y_pred)
global_rmse = RMSE(y_true, y_pred)
global_r    = np.corrcoef(y_true, y_pred)[0,1]
global_r2   = global_r**2
global_mape = MAPE(y_true, y_pred)

print("\n=== GLOBAL PERFORMANCE (GP-age) ===")
print(f"MAE:  {global_mae:.3f}")
print(f"RMSE: {global_rmse:.3f}")
print(f"R:    {global_r:.3f}")
print(f"R²:   {global_r2:.3f}")
print(f"MAPE: {global_mape:.3f}%")

# ------------------------------------------------------------
# 7. GROUP PERFORMANCE
# ------------------------------------------------------------

print("\n=== GROUP PERFORMANCE ===")
for grp in [0,1]:
    sub = eval_df[eval_df["label"]==grp]
    if len(sub)==0: continue
    yt, yp = sub["age_at_surgery"], sub["GP_age"]
    print(f"\nGroup {grp} — n={len(sub)}")
    print(f"  MAE:  {MAE(yt,yp):.3f}")
    print(f"  RMSE: {RMSE(yt,yp):.3f}")
    print(f"  R:    {np.corrcoef(yt,yp)[0,1]:.3f}")

# ------------------------------------------------------------
# 8. AGE ACCELERATION
# ------------------------------------------------------------

eval_df["age_acc"] = eval_df["GP_age"] - eval_df["age_at_surgery"]

print("\n=== AGE ACCELERATION SUMMARY ===")
print(eval_df["age_acc"].describe())

# ------------------------------------------------------------
# 9. FINAL SUMMARY
# ------------------------------------------------------------

print("\n======================================================")
print("               GP-AGE — FINAL SUMMARY")
print("======================================================")
print(f"Samples analysed:   {len(eval_df)}")
print(f"CpGs required:      {len(gp_age_cpgs)}")
print(f"CpGs present:       {len(gp_age_cpgs)-len(missing_cpgs)}")
print(f"CpGs missing:       {len(missing_cpgs)}")
print("------------------------------------------------------")
print(f"MAE: {global_mae:.2f} years")
print(f"R²:  {global_r2:.3f}")
print("======================================================\n")


GP-age CpGs required: 30

=== CPG COVERAGE ===
Required: 30
Present : 25
Missing : 5

=== CONTENT OF GP-age PREDICTION FILE ===
       sample  predictions
0  GSM7057528       44.419
1  GSM7057538       53.156
2  GSM7057546       49.298
3  GSM7057554       39.014
4  GSM7057559       44.281
Columns: ['sample', 'predictions']


ValueError: No column containing 'age' found in prediction file.

In [5]:
# ------------------------------------------------------------
# 0. IDENTIFY GP-age prediction and stats files
# ------------------------------------------------------------
import pandas as pd
from pathlib import Path

pred_file = Path("./gp_age_run_GSE225845/gp_age_output/GP-age_30_cpgs_predictions.csv")
stats_file = Path("./gp_age_run_GSE225845/gp_age_output/GP-age_30_cpgs_stats.csv")

if not pred_file.exists():
    raise FileNotFoundError(f"Prediction file not found: {pred_file}")

print("Using prediction file:", pred_file)

# ------------------------------------------------------------
# 1. LOAD PREDICTIONS (sample_id + predicted_age)
# ------------------------------------------------------------
gp_pred_df = pd.read_csv(pred_file)

print("\n=== PREDICTION FILE HEAD ===")
print(gp_pred_df.head())

# Detect the correct prediction column
pred_cols = [c for c in gp_pred_df.columns if "pred" in c.lower() or "age" in c.lower()]

if len(pred_cols) == 0:
    raise ValueError(
        "No predicted age column found in prediction file. "
        f"Columns available: {gp_pred_df.columns.tolist()}"
    )

pred_age_col = pred_cols[-1]
print("Using predicted age column:", pred_age_col)

# Must contain sample IDs
id_cols = [c for c in gp_pred_df.columns if "id" in c.lower()]
if len(id_cols) == 0:
    raise ValueError(
        "No column containing sample ID was found. "
        f"Columns available: {gp_pred_df.columns.tolist()}"
    )

sample_id_col = id_cols[0]
print("Using sample ID column:", sample_id_col)

# Extract final aligned vector
gp_pred_age = gp_pred_df.set_index(sample_id_col)[pred_age_col]
gp_pred_age = gp_pred_age.astype(float)


Using prediction file: gp_age_run_GSE225845/gp_age_output/GP-age_30_cpgs_predictions.csv

=== PREDICTION FILE HEAD ===
       sample  predictions
0  GSM7057528       44.419
1  GSM7057538       53.156
2  GSM7057546       49.298
3  GSM7057554       39.014
4  GSM7057559       44.281
Using predicted age column: predictions


ValueError: No column containing sample ID was found. Columns available: ['sample', 'predictions']

In [6]:
# ============================================================
# LOAD GP-age predictions correctly (Kaggle version)
# ============================================================

import pandas as pd
from pathlib import Path

pred_file = Path("./gp_age_run_GSE225845/gp_age_output/GP-age_30_cpgs_predictions.csv")

if not pred_file.exists():
    raise FileNotFoundError(f"Prediction file not found: {pred_file}")

print("Using prediction file:", pred_file)

# Load CSV
gp_pred_df = pd.read_csv(pred_file)

print("\n=== HEAD OF PREDICTION FILE ===")
print(gp_pred_df.head())

# ------------------------------------------------------------
# The file contains exactly two columns:
#   "sample"       → sample ID
#   "predictions"  → predicted age
# ------------------------------------------------------------

if "sample" not in gp_pred_df.columns:
    raise ValueError("Column 'sample' not found in prediction file.")

if "predictions" not in gp_pred_df.columns:
    raise ValueError("Column 'predictions' not found in prediction file.")

# Build Series indexed by sample ID
gp_pred_age = (
    gp_pred_df
    .set_index("sample")["predictions"]
    .astype(float)
)

print("\n✓ Successfully extracted GP-age predictions.")
print("Number of predictions:", len(gp_pred_age))
print("First few predictions:")
print(gp_pred_age.head())


Using prediction file: gp_age_run_GSE225845/gp_age_output/GP-age_30_cpgs_predictions.csv

=== HEAD OF PREDICTION FILE ===
       sample  predictions
0  GSM7057528       44.419
1  GSM7057538       53.156
2  GSM7057546       49.298
3  GSM7057554       39.014
4  GSM7057559       44.281

✓ Successfully extracted GP-age predictions.
Number of predictions: 253
First few predictions:
sample
GSM7057528    44.419
GSM7057538    53.156
GSM7057546    49.298
GSM7057554    39.014
GSM7057559    44.281
Name: predictions, dtype: float64


In [7]:
# ============================================================
# GP-AGE — CpG coverage + accuracy metrics (GSE225845)
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

MODEL_TYPE = "30"  # deve essere lo stesso usato nel run di GP-age

# ------------------------------------------------------------
# 1. CpG coverage: quanti CpG del modello sono nel tuo dataset?
# ------------------------------------------------------------
model_file = Path("./GP-age/model_data") / f"GP-age_sites_{MODEL_TYPE}_cpgs.csv"
if not model_file.exists():
    raise FileNotFoundError(f"Required GP-age CpG list not found: {model_file}")

gp_age_cpgs = pd.read_csv(model_file).iloc[:, 0].tolist()
print(f"GP-age clock CpGs required for model {MODEL_TYPE}: {len(gp_age_cpgs)}")

# Methylation matrix usata per GP-age (CpG in righe, sample in colonne)
METH_CSV_PATH = Path("./gp_age_run_GSE225845/gp_age_meth_GSE225845_normal_adj.csv")
if not METH_CSV_PATH.exists():
    raise FileNotFoundError(f"Methylation CSV not found: {METH_CSV_PATH}")

meth_raw = pd.read_csv(METH_CSV_PATH, index_col=0)
present_cpgs = set(meth_raw.index)

missing_cpgs = [c for c in gp_age_cpgs if c not in present_cpgs]
present = len(gp_age_cpgs) - len(missing_cpgs)

print("\n=== CPG COVERAGE REPORT ===")
print(f"CpGs required by GP-age: {len(gp_age_cpgs)}")
print(f"CpGs found in dataset:   {present}")
print(f"CpGs missing:            {len(missing_cpgs)}")
if missing_cpgs:
    print("Missing CpGs (first 10):", missing_cpgs[:10])

# ------------------------------------------------------------
# 2. Carica le predizioni GP-age dal CSV di output
# ------------------------------------------------------------
pred_file = Path("./gp_age_run_GSE225845/gp_age_output/GP-age_30_cpgs_predictions.csv")
if not pred_file.exists():
    raise FileNotFoundError(f"Prediction file not found: {pred_file}")

gp_pred_df = pd.read_csv(pred_file)

if "sample" not in gp_pred_df.columns or "predictions" not in gp_pred_df.columns:
    raise ValueError(
        "Prediction CSV must contain columns 'sample' and 'predictions'. "
        f"Found columns: {gp_pred_df.columns.tolist()}"
    )

gp_pred_age = (
    gp_pred_df
    .set_index("sample")["predictions"]
    .astype(float)
)

print("\n>>> GP-age predictions loaded:")
print(gp_pred_age.head())

# ------------------------------------------------------------
# 3. Costruisci il dataframe di valutazione (pheno + GP_age)
# ------------------------------------------------------------
# pheno_pl è un DataFrame Polars creato prima
pheno = pheno_pl.to_pandas().copy()

# mappa le predizioni sulle righe di pheno usando SAMPLE_ID_COL (es. 'id_tissue')
pheno["GP_age"] = pheno[SAMPLE_ID_COL].map(gp_pred_age)

# tieni solo Normal + Adjacent (0,1) con entrambe le età presenti
eval_df = (
    pheno[pheno[LABEL_COL].isin([0, 1])]
    .dropna(subset=[AGE_COL, "GP_age"])
    .copy()
)

print("\nSamples used for evaluation:", len(eval_df))

y_true = eval_df[AGE_COL].values
y_pred = eval_df["GP_age"].values

# ------------------------------------------------------------
# 4. Funzioni metriche
# ------------------------------------------------------------
def RMSE(y, yhat): return np.sqrt(np.mean((y - yhat) ** 2))
def MAE(y, yhat):  return np.mean(np.abs(y - yhat))
def MAPE(y, yhat): return np.mean(np.abs((y - yhat) / y)) * 100

global_mae  = MAE(y_true, y_pred)
global_rmse = RMSE(y_true, y_pred)
global_r    = np.corrcoef(y_true, y_pred)[0, 1]
global_r2   = global_r ** 2
global_mape = MAPE(y_true, y_pred)

print("\n=== GLOBAL PERFORMANCE (GP-age, Normal+Adjacent) ===")
print(f"N samples:      {len(eval_df)}")
print(f"MAE   (years):  {global_mae:.3f}")
print(f"RMSE  (years):  {global_rmse:.3f}")
print(f"R     (corr):   {global_r:.3f}")
print(f"R²:             {global_r2:.3f}")
print(f"MAPE   (%):     {global_mape:.3f}")

# ------------------------------------------------------------
# 5. Metriche per gruppo (0=Normal, 1=Adjacent)
# ------------------------------------------------------------
print("\n=== GROUP-SPECIFIC PERFORMANCE ===")
for grp in [0, 1]:
    subset = eval_df[eval_df[LABEL_COL] == grp]
    if len(subset) == 0:
        continue
    yt = subset[AGE_COL].values
    yp = subset["GP_age"].values
    r  = np.corrcoef(yt, yp)[0, 1]
    print(f"\nGroup {grp} ({'Normal' if grp == 0 else 'Adjacent'}) — n={len(subset)}")
    print(f"  MAE:   {MAE(yt, yp):.3f}")
    print(f"  RMSE:  {RMSE(yt, yp):.3f}")
    print(f"  R:     {r:.3f}")
    print(f"  MAPE:  {MAPE(yt, yp):.2f}%")

# ------------------------------------------------------------
# 6. Age acceleration
# ------------------------------------------------------------
eval_df["age_acc"] = eval_df["GP_age"] - eval_df[AGE_COL]

print("\n=== AGE ACCELERATION SUMMARY ===")
print(eval_df["age_acc"].describe())

print("\n=== AGE ACCELERATION BY GROUP ===")
print(eval_df.groupby(LABEL_COL)["age_acc"].describe())

# ------------------------------------------------------------
# 7. Final compact summary (Horvath-style)
# ------------------------------------------------------------
print("\n======================================================")
print("            GP-AGE — FINAL SUMMARY REPORT")
print("======================================================")
print(f"Samples analysed (Normal+Adjacent): {len(eval_df)}")
print(f"CpGs required by GP-age:            {len(gp_age_cpgs)}")
print(f"CpGs present in dataset:            {present}")
print(f"CpGs missing:                       {len(missing_cpgs)}")
print("------------------------------------------------------")
print(f"MAE: {global_mae:.2f} years")
print(f"RMSE: {global_rmse:.2f} years")
print(f"R²:  {global_r2:.3f}")
print("======================================================\n")


GP-age clock CpGs required for model 30: 30

=== CPG COVERAGE REPORT ===
CpGs required by GP-age: 30
CpGs found in dataset:   25
CpGs missing:            5
Missing CpGs (first 10): ['cg24079702', 'cg08128734', 'cg00329615', 'cg23479922', 'cg25413977']

>>> GP-age predictions loaded:
sample
GSM7057528    44.419
GSM7057538    53.156
GSM7057546    49.298
GSM7057554    39.014
GSM7057559    44.281
Name: predictions, dtype: float64

Samples used for evaluation: 253

=== GLOBAL PERFORMANCE (GP-age, Normal+Adjacent) ===
N samples:      253
MAE   (years):  7.775
RMSE  (years):  10.654
R     (corr):   0.719
R²:             0.517
MAPE   (%):     14.218

=== GROUP-SPECIFIC PERFORMANCE ===

Group 0 (Normal) — n=113
  MAE:   4.985
  RMSE:  6.591
  R:     0.670
  MAPE:  11.47%

Group 1 (Adjacent) — n=140
  MAE:   10.027
  RMSE:  13.040
  R:     0.623
  MAPE:  16.44%

=== AGE ACCELERATION SUMMARY ===
count    253.000000
mean      -4.313071
std        9.761124
min      -36.716000
25%       -9.999000
50